# HEAL MATRIX: AI-Powered Multi-Agent Healthcare Platform
## Step-by-Step Implementation Guide

This notebook provides a structured, cell-by-cell implementation of the HEAL MATRIX system.

**Author:** HEAL MATRIX Development Team  
**Platform:** Google Colab  
**Runtime:** Python 3.10+ with GPU

---
## STEP 1: Mount Google Drive
Mount your Google Drive to save files and access data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

---
## STEP 2: Install Core Dependencies
Install essential packages for LangChain, LangGraph, and FastAPI.

In [ ]:
print("📦 Installing LangChain and LangGraph...")
!pip install -q langchain langchain-openai langgraph
print("✅ LangChain packages installed!")

---
## STEP 3: Install Web Framework Dependencies
Install FastAPI, Uvicorn, and Gradio for the web interface.

In [ ]:
print("📦 Installing web frameworks...")
!pip install -q fastapi uvicorn gradio
print("✅ Web frameworks installed!")

---
## STEP 4: Install Additional Utilities
Install Twilio, Pydantic, Requests, and async utilities.

In [ ]:
print("📦 Installing utilities...")
!pip install -q twilio pydantic requests nest-asyncio pyngrok
print("✅ Utility packages installed!")

---
## STEP 5: Install Voice and Location Services
Install Google Text-to-Speech and Google Maps API.

In [ ]:
print("📦 Installing TTS and Maps services...")
!pip install -q gtts googlemaps
print("✅ Voice and location services installed!")

---
## STEP 6: Install Computer Vision Libraries
Install OpenCV, PIL, Transformers, PyTorch for image processing.

In [ ]:
print("📦 Installing computer vision libraries...")
!pip install -q pillow opencv-python-headless transformers torch torchvision
print("✅ Computer vision packages installed!")

---
## STEP 7: Install OpenAI SDK
Install OpenAI package for LLM integration.

In [ ]:
print("📦 Installing OpenAI SDK...")
!pip install -q openai
print("✅ OpenAI package installed!")

---
## STEP 8: Install Ollama
Download and install Ollama for local LLM hosting.

In [ ]:
print("🔧 Installing Ollama...")
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
print("✅ Ollama installed successfully!")

---
## STEP 9: Start Ollama Server
Launch Ollama server in background.

In [ ]:
print("🚀 Starting Ollama server...")
import subprocess
import time

# Start Ollama server in background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("✅ Ollama server running!")

---
## STEP 10: Download Gemma 2B Model
Pull the Gemma 2B model from Ollama.

In [ ]:
print("📥 Downloading gemma2:2b model... (this may take a few minutes)")
!ollama pull gemma2:2b
print("✅ Model downloaded successfully!")

---
## STEP 11: Import Required Libraries
Import all necessary Python libraries for the project.

In [ ]:
# Core imports
import os
import json
import time
from typing import TypedDict, Annotated, List, Dict, Any
from datetime import datetime

# LangChain and LangGraph
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END

# Web and API
import gradio as gr
import requests
import googlemaps

# Computer Vision
from PIL import Image
import cv2
import numpy as np
from transformers import CLIPProcessor, CLIPModel

# Voice
from gtts import gTTS

print("✅ All libraries imported successfully!")

---
## STEP 12: Set API Keys
Configure API keys for Google Maps and other services.

In [ ]:
# Set your API keys here
GOOGLE_MAPS_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY_HERE"

# Initialize Google Maps client
gmaps = googlemaps.Client(key=GOOGLE_MAPS_API_KEY)

print("✅ API keys configured!")
print("⚠️  Remember to replace with your actual API keys!")

---
## STEP 13: Initialize Ollama LLM
Create ChatOpenAI instance pointing to local Ollama server.

In [ ]:
# Initialize Ollama LLM
llm = ChatOpenAI(
    base_url="http://localhost:11434/v1",
    model="gemma2:2b",
    api_key="ollama",  # Dummy key for Ollama
    temperature=0.7,
    max_tokens=512
)

print("✅ Ollama LLM initialized!")
print(f"   Model: gemma2:2b")
print(f"   Base URL: http://localhost:11434/v1")

---
## STEP 14: Initialize CLIP Model
Load CLIP model for computer vision tasks.

In [ ]:
print("📥 Loading CLIP model...")

# Load CLIP model and processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("✅ CLIP model loaded successfully!")

---
## STEP 15: Define State Schema
Create the state structure for LangGraph.

In [ ]:
class AgentState(TypedDict):
    """State schema for the multi-agent system"""
    messages: List[Dict[str, str]]  # Conversation history
    current_agent: str  # Current active agent
    user_input: str  # Latest user input
    image_path: str  # Path to uploaded image
    image_description: str  # Description from CLIP
    location: str  # User location
    context: Dict[str, Any]  # Additional context
    final_response: str  # Final response to user

print("✅ State schema defined!")

---
## STEP 16: Create Image Analysis Function
Function to analyze medical images using CLIP.

In [ ]:
def analyze_image(image_path: str) -> str:
    """
    Analyze medical image using CLIP model
    
    Args:
        image_path: Path to the image file
    
    Returns:
        Description of the image
    """
    try:
        # Load image
        image = Image.open(image_path).convert("RGB")
        
        # Medical-related labels for zero-shot classification
        labels = [
            "healthy skin",
            "skin rash",
            "wound or injury",
            "bruising",
            "swelling",
            "infection",
            "medical test result",
            "x-ray or scan",
            "medical document"
        ]
        
        # Process image and labels
        inputs = clip_processor(
            text=labels,
            images=image,
            return_tensors="pt",
            padding=True
        )
        
        # Get predictions
        outputs = clip_model(**inputs)
        logits = outputs.logits_per_image[0]
        probs = logits.softmax(dim=0)
        
        # Get top 3 predictions
        top_3_idx = probs.argsort(descending=True)[:3]
        
        description = "Image Analysis Results:\n"
        for idx in top_3_idx:
            description += f"- {labels[idx]}: {probs[idx].item():.2%}\n"
        
        return description
        
    except Exception as e:
        return f"Error analyzing image: {str(e)}"

print("✅ Image analysis function created!")

---
## STEP 17: Create Google Maps Functions
Functions for finding nearby medical facilities.

In [ ]:
def find_nearby_facilities(location: str, facility_type: str = "hospital") -> str:
    """
    Find nearby medical facilities using Google Maps API
    
    Args:
        location: User's location (address or coordinates)
        facility_type: Type of facility (hospital, pharmacy, clinic)
    
    Returns:
        Formatted list of nearby facilities
    """
    try:
        # Geocode the location
        geocode_result = gmaps.geocode(location)
        if not geocode_result:
            return "Could not find the specified location."
        
        lat = geocode_result[0]['geometry']['location']['lat']
        lng = geocode_result[0]['geometry']['location']['lng']
        
        # Search for nearby facilities
        places_result = gmaps.places_nearby(
            location=(lat, lng),
            radius=5000,  # 5km radius
            type=facility_type
        )
        
        if not places_result.get('results'):
            return f"No {facility_type}s found nearby."
        
        # Format results
        facilities = []
        for i, place in enumerate(places_result['results'][:5], 1):
            name = place.get('name', 'Unknown')
            address = place.get('vicinity', 'Address not available')
            rating = place.get('rating', 'N/A')
            
            facilities.append(
                f"{i}. {name}\n   Address: {address}\n   Rating: {rating}/5"
            )
        
        return "\n\n".join(facilities)
        
    except Exception as e:
        return f"Error finding facilities: {str(e)}"

print("✅ Google Maps functions created!")

---
## STEP 18: Create Router Agent
Agent to route user queries to appropriate specialized agents.

In [ ]:
def router_agent(state: AgentState) -> AgentState:
    """
    Router agent - analyzes user input and routes to appropriate agent
    """
    user_input = state['user_input'].lower()
    
    # Emergency keywords
    if any(word in user_input for word in ['emergency', 'urgent', 'help', 'accident', 'bleeding', 'pain']):
        state['current_agent'] = 'emergency'
    
    # Appointment keywords
    elif any(word in user_input for word in ['appointment', 'book', 'schedule', 'doctor', 'visit']):
        state['current_agent'] = 'appointment'
    
    # Prescription keywords
    elif any(word in user_input for word in ['prescription', 'medicine', 'medication', 'pharmacy', 'drug']):
        state['current_agent'] = 'prescription'
    
    # Health monitoring keywords
    elif any(word in user_input for word in ['track', 'monitor', 'health', 'vitals', 'record']):
        state['current_agent'] = 'health_monitor'
    
    # Medical consultation (default)
    else:
        state['current_agent'] = 'medical_consultation'
    
    state['messages'].append({
        'role': 'system',
        'content': f"Routing to {state['current_agent']} agent"
    })
    
    return state

print("✅ Router agent created!")

---
## STEP 19: Create Emergency Response Agent
Agent to handle medical emergencies.

In [ ]:
def emergency_agent(state: AgentState) -> AgentState:
    """
    Emergency response agent - handles urgent medical situations
    """
    system_prompt = """
    You are an emergency medical response AI assistant. 
    Provide immediate, clear, and actionable first-aid advice.
    Always recommend calling emergency services (911) for serious conditions.
    Be calm, direct, and prioritize safety.
    """
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=state['user_input'])
    ]
    
    # Get LLM response
    response = llm.invoke(messages)
    ai_response = response.content
    
    # Find nearby emergency facilities
    if state.get('location'):
        facilities = find_nearby_facilities(state['location'], 'hospital')
        ai_response += f"\n\n📍 Nearest Emergency Facilities:\n{facilities}"
    
    state['final_response'] = ai_response
    state['messages'].append({
        'role': 'assistant',
        'content': ai_response
    })
    
    return state

print("✅ Emergency agent created!")

---
## STEP 20: Create Medical Consultation Agent
Agent for general medical advice and consultations.

In [ ]:
def medical_consultation_agent(state: AgentState) -> AgentState:
    """
    Medical consultation agent - provides medical advice and symptom analysis
    """
    system_prompt = """
    You are a medical consultation AI assistant.
    Provide helpful health information and symptom analysis.
    Always include appropriate disclaimers that you are not replacing professional medical advice.
    Be empathetic, thorough, and evidence-based.
    """
    
    user_message = state['user_input']
    
    # Add image analysis if available
    if state.get('image_description'):
        user_message += f"\n\nImage Analysis: {state['image_description']}"
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_message)
    ]
    
    # Get LLM response
    response = llm.invoke(messages)
    ai_response = response.content
    
    # Add disclaimer
    ai_response += "\n\n⚠️ Disclaimer: This is informational only. Please consult a healthcare professional for medical advice."
    
    state['final_response'] = ai_response
    state['messages'].append({
        'role': 'assistant',
        'content': ai_response
    })
    
    return state

print("✅ Medical consultation agent created!")

---
## STEP 21: Create Appointment Scheduling Agent
Agent to manage appointment bookings.

In [ ]:
def appointment_agent(state: AgentState) -> AgentState:
    """
    Appointment scheduling agent - handles appointment bookings
    """
    system_prompt = """
    You are an appointment scheduling assistant.
    Help users book, modify, or cancel medical appointments.
    Ask for necessary details: date, time, doctor specialty, reason for visit.
    Be efficient and friendly.
    """
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=state['user_input'])
    ]
    
    # Get LLM response
    response = llm.invoke(messages)
    ai_response = response.content
    
    # Simulated appointment confirmation
    ai_response += "\n\n📅 Your appointment request has been noted. You will receive a confirmation shortly."
    
    state['final_response'] = ai_response
    state['messages'].append({
        'role': 'assistant',
        'content': ai_response
    })
    
    return state

print("✅ Appointment agent created!")

---
## STEP 22: Create Prescription Management Agent
Agent to handle prescription-related queries.

In [ ]:
def prescription_agent(state: AgentState) -> AgentState:
    """
    Prescription management agent - handles medication information
    """
    system_prompt = """
    You are a prescription management assistant.
    Provide information about medications, dosages, and side effects.
    Help users find nearby pharmacies.
    Always remind users to follow their doctor's prescription.
    """
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=state['user_input'])
    ]
    
    # Get LLM response
    response = llm.invoke(messages)
    ai_response = response.content
    
    # Find nearby pharmacies
    if state.get('location'):
        pharmacies = find_nearby_facilities(state['location'], 'pharmacy')
        ai_response += f"\n\n💊 Nearby Pharmacies:\n{pharmacies}"
    
    state['final_response'] = ai_response
    state['messages'].append({
        'role': 'assistant',
        'content': ai_response
    })
    
    return state

print("✅ Prescription agent created!")

---
## STEP 23: Create Health Monitoring Agent
Agent to track health metrics and vitals.

In [ ]:
def health_monitor_agent(state: AgentState) -> AgentState:
    """
    Health monitoring agent - tracks health metrics
    """
    system_prompt = """
    You are a health monitoring assistant.
    Help users track their health metrics, vital signs, and wellness data.
    Provide insights and trends based on their data.
    Encourage healthy habits and alert to concerning patterns.
    """
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=state['user_input'])
    ]
    
    # Get LLM response
    response = llm.invoke(messages)
    ai_response = response.content
    
    ai_response += "\n\n📊 Health data logged. Continue tracking for better insights!"
    
    state['final_response'] = ai_response
    state['messages'].append({
        'role': 'assistant',
        'content': ai_response
    })
    
    return state

print("✅ Health monitoring agent created!")

---
## STEP 24: Create Text-to-Speech Function
Function to convert text responses to speech.

In [ ]:
def text_to_speech(text: str, output_path: str = "response.mp3") -> str:
    """
    Convert text to speech using Google TTS
    
    Args:
        text: Text to convert
        output_path: Output file path
    
    Returns:
        Path to audio file
    """
    try:
        tts = gTTS(text=text, lang='en', slow=False)
        tts.save(output_path)
        return output_path
    except Exception as e:
        print(f"TTS Error: {e}")
        return None

print("✅ Text-to-speech function created!")

---
## STEP 25: Build LangGraph Workflow
Create the state machine with all agents.

In [ ]:
# Create the graph
workflow = StateGraph(AgentState)

# Add nodes (agents)
workflow.add_node("router", router_agent)
workflow.add_node("emergency", emergency_agent)
workflow.add_node("medical_consultation", medical_consultation_agent)
workflow.add_node("appointment", appointment_agent)
workflow.add_node("prescription", prescription_agent)
workflow.add_node("health_monitor", health_monitor_agent)

# Set entry point
workflow.set_entry_point("router")

# Add conditional edges from router to specialized agents
def route_to_agent(state: AgentState) -> str:
    return state['current_agent']

workflow.add_conditional_edges(
    "router",
    route_to_agent,
    {
        "emergency": "emergency",
        "medical_consultation": "medical_consultation",
        "appointment": "appointment",
        "prescription": "prescription",
        "health_monitor": "health_monitor"
    }
)

# All agents lead to END
workflow.add_edge("emergency", END)
workflow.add_edge("medical_consultation", END)
workflow.add_edge("appointment", END)
workflow.add_edge("prescription", END)
workflow.add_edge("health_monitor", END)

# Compile the graph
app = workflow.compile()

print("✅ LangGraph workflow built and compiled!")

---
## STEP 26: Create Main Processing Function
Function to process user queries through the agent system.

In [ ]:
def process_query(user_input: str, image_path: str = None, location: str = "New York, NY"):
    """
    Process user query through the multi-agent system
    
    Args:
        user_input: User's text input
        image_path: Optional path to uploaded image
        location: User's location
    
    Returns:
        Tuple of (text_response, audio_path)
    """
    # Initialize state
    initial_state = {
        'messages': [],
        'current_agent': '',
        'user_input': user_input,
        'image_path': image_path or '',
        'image_description': '',
        'location': location,
        'context': {},
        'final_response': ''
    }
    
    # Analyze image if provided
    if image_path:
        initial_state['image_description'] = analyze_image(image_path)
    
    # Run the workflow
    result = app.invoke(initial_state)
    
    # Get final response
    text_response = result['final_response']
    
    # Generate audio
    audio_path = text_to_speech(text_response)
    
    return text_response, audio_path

print("✅ Main processing function created!")

---
## STEP 27: Create Gradio Interface
Build the web UI for user interaction.

In [ ]:
def gradio_interface(user_input, image, location):
    """
    Gradio interface function
    """
    if not user_input:
        return "Please enter a query.", None
    
    # Process the query
    image_path = image if image else None
    text_response, audio_path = process_query(
        user_input=user_input,
        image_path=image_path,
        location=location
    )
    
    return text_response, audio_path

# Create Gradio interface
iface = gr.Interface(
    fn=gradio_interface,
    inputs=[
        gr.Textbox(
            label="Your Query",
            placeholder="Describe your symptoms, ask a medical question, or request an appointment...",
            lines=3
        ),
        gr.Image(
            label="Upload Medical Image (Optional)",
            type="filepath"
        ),
        gr.Textbox(
            label="Your Location",
            value="New York, NY",
            placeholder="Enter your city and state"
        )
    ],
    outputs=[
        gr.Textbox(label="Response", lines=10),
        gr.Audio(label="Voice Response")
    ],
    title="🏥 HEAL MATRIX - AI Healthcare Assistant",
    description="""
    Multi-agent AI system for healthcare management.
    
    **Features:**
    - 🚨 Emergency Response
    - 💬 Medical Consultation
    - 📅 Appointment Scheduling
    - 💊 Prescription Management
    - 📊 Health Monitoring
    - 🖼️ Medical Image Analysis
    - 🔊 Voice Responses
    
    ⚠️ **Disclaimer:** This is an AI assistant for informational purposes only. 
    Always consult healthcare professionals for medical advice.
    """,
    theme="soft",
    examples=[
        ["I have a severe headache and feel dizzy", None, "New York, NY"],
        ["I need to book an appointment with a cardiologist", None, "Los Angeles, CA"],
        ["What are the side effects of aspirin?", None, "Chicago, IL"],
        ["Help! I think I'm having an allergic reaction", None, "Houston, TX"]
    ]
)

print("✅ Gradio interface created!")

---
## STEP 28: Launch Gradio App
Start the Gradio web server and create public link.

In [ ]:
# Launch the Gradio app
print("🚀 Launching HEAL MATRIX...")
print("\n" + "="*60)
print("HEAL MATRIX is starting...")
print("="*60 + "\n")

# Launch with public link
iface.launch(
    share=True,
    debug=True,
    show_error=True
)

print("\n✅ HEAL MATRIX is now running!")
print("📱 Access the interface using the public URL above")

---
## STEP 29: Test the System (Optional)
Run basic tests to verify functionality.

In [ ]:
# Test emergency query
print("Testing Emergency Agent...")
response, audio = process_query(
    "I'm having chest pain",
    location="New York, NY"
)
print(f"Response: {response[:200]}...")
print("\n" + "="*60 + "\n")

# Test medical consultation
print("Testing Medical Consultation Agent...")
response, audio = process_query(
    "What are the symptoms of flu?",
    location="New York, NY"
)
print(f"Response: {response[:200]}...")
print("\n" + "="*60 + "\n")

# Test appointment booking
print("Testing Appointment Agent...")
response, audio = process_query(
    "I need to book a dental appointment",
    location="New York, NY"
)
print(f"Response: {response[:200]}...")

print("\n✅ All tests completed!")

---
## STEP 30: System Information
Display system configuration and status.

In [ ]:
print("="*60)
print("HEAL MATRIX - SYSTEM INFORMATION")
print("="*60)
print(f"\n🤖 LLM Model: Ollama Gemma 2B")
print(f"🔗 LLM Endpoint: http://localhost:11434/v1")
print(f"👁️ Vision Model: CLIP ViT-B/32")
print(f"🗣️ TTS Engine: Google Text-to-Speech")
print(f"📍 Location Service: Google Maps API")
print(f"🌐 Web Framework: Gradio")
print(f"🧠 Agent Framework: LangGraph")
print(f"\n👥 Available Agents:")
print("   1. Router Agent (Intent Classification)")
print("   2. Emergency Response Agent")
print("   3. Medical Consultation Agent")
print("   4. Appointment Scheduling Agent")
print("   5. Prescription Management Agent")
print("   6. Health Monitoring Agent")
print(f"\n✅ System Status: OPERATIONAL")
print("="*60)